<a href="https://colab.research.google.com/github/nadayali710-spec/Programming-For-AI-Project/blob/main/Hockey_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Shared helpers for the Hockey Team Performance project.

Contains: paths, chronological split, custom transformers, a leakage-safe
preprocessing pipeline builder, and metric helpers.
"""
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, r2_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
ROOT = Path(os.environ.get("HOCKEY_PROJECT_DIR", Path.cwd()))
DATA, FIG, RES = ROOT / "data", ROOT / "figures", ROOT / "results"
for _p in (DATA, FIG, RES):
    _p.mkdir(parents=True, exist_ok=True)

RAW_CSV = DATA / "hockey_raw.csv"
RAW_BACKUP = DATA / "hockey_raw_UNCHANGED_COPY.csv"
RAW_META = DATA / "hockey_raw_metadata.json"
WORKING_CSV = DATA / "hockey_working_5pct_missing.csv"
CLEAN_CSV = DATA / "hockey_clean_eda.csv"

# Model inputs (Task A and Task B). Wins, Win %, Losses, OT Losses are excluded.
CAT = ["Team Name"]
NUM = ["Year", "GF", "GA", "+/-"]
FEATURES = CAT + NUM


# --------------------------------------------------------------------------
# Data loading and splitting
# --------------------------------------------------------------------------
def load_working():
    """Load the working copy (with controlled missing values), drop exact duplicates."""
    df = pd.read_csv(WORKING_CSV)
    return df.drop_duplicates().reset_index(drop=True)


def chronological_split(df, test_frac=0.2):
    """Train on earlier seasons, test on later ones.

    The cut is placed at a season boundary so that the training share is as
    close as possible to 1 - test_frac (whole seasons are never split).
    Returns (train, test, last_train_year).
    """
    df = df.sort_values("Year", kind="stable").reset_index(drop=True)
    share = df.groupby("Year").size().cumsum() / len(df)
    cutoff = (share - (1 - test_frac)).abs().idxmin()
    return df[df["Year"] <= cutoff].copy(), df[df["Year"] > cutoff].copy(), int(cutoff)


def skewed_columns(train, cols, threshold=0.5):
    """Columns suited to log1p: meaningfully RIGHT-skewed (skew > threshold) and
    non-negative. log1p compresses a long right tail; it would make a left-skewed
    column worse. Decided on TRAINING data only."""
    out = []
    for c in cols:
        s = train[c].dropna()
        if s.skew() > threshold and s.min() >= 0:
            out.append(c)
    return out


# --------------------------------------------------------------------------
# Custom transformers (all fitted on training data only inside a Pipeline)
# --------------------------------------------------------------------------
class PythagoreanExpectation(BaseEstimator, TransformerMixin):
    """Hockey Pythagorean expectation: GF^2 / (GF^2 + GA^2)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        gf2, ga2 = X["GF"] ** 2, X["GA"] ** 2
        denom = (gf2 + ga2).replace(0, np.nan)
        X["Pythagorean"] = (gf2 / denom).fillna(0.5)
        return X


class IQRCapper(BaseEstimator, TransformerMixin):
    """Clip columns to [Q1 - k*IQR, Q3 + k*IQR]; bounds learned on train."""

    def __init__(self, cols, k=1.5):
        self.cols, self.k = cols, k

    def fit(self, X, y=None):
        q1, q3 = X[self.cols].quantile(0.25), X[self.cols].quantile(0.75)
        iqr = q3 - q1
        self.lower_, self.upper_ = q1 - self.k * iqr, q3 + self.k * iqr
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = X[self.cols].clip(self.lower_, self.upper_, axis=1)
        return X


class Log1pColumns(BaseEstimator, TransformerMixin):
    """log1p on selected non-negative columns (negatives clipped to 0 as a guard)."""

    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.cols] = np.log1p(X[self.cols].clip(lower=0))
        return X


class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.cols)


def build_preprocessor(use_pyth=False, cap=False, log_cols=(), drop_cols=(), scale=True):
    """Leakage-safe preprocessing pipeline.

    Order: impute -> (Pythagorean) -> (IQR cap) -> (log1p) -> (drop) ->
           StandardScaler on numerics + OneHotEncoder(handle_unknown="ignore") on Team Name.
    Every step is fitted on the training fold only when used inside a Pipeline.
    """
    imputer = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), NUM),
         ("cat", SimpleImputer(strategy="most_frequent"), CAT)],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    steps = [("impute", imputer)]
    if use_pyth:
        steps.append(("pyth", PythagoreanExpectation()))
    if cap:
        steps.append(("cap", IQRCapper([c for c in NUM if c != "Year"])))
    if log_cols:
        steps.append(("log", Log1pColumns(list(log_cols))))
    if drop_cols:
        steps.append(("drop", DropColumns(list(drop_cols))))

    num_cols = [c for c in NUM + (["Pythagorean"] if use_pyth else []) if c not in drop_cols]
    final = ColumnTransformer(
        [("num", StandardScaler() if scale else "passthrough", num_cols),
         ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)]
    ).set_output(transform="pandas")
    steps.append(("encode_scale", final))
    return Pipeline(steps)


def make_pipeline(estimator, **prep_kwargs):
    return Pipeline([("prep", build_preprocessor(**prep_kwargs)), ("model", estimator)])


# --------------------------------------------------------------------------
# Metrics
# --------------------------------------------------------------------------
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": float(np.sqrt(mse)), "R2": r2_score(y_true, y_pred)}


def classification_metrics(y_true, y_pred, score):
    """`score` = predicted probability (or decision score) of the positive class."""
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, score)}


def md_table(df, floatfmt=".3f"):
    """Minimal DataFrame -> Markdown table (no extra dependency)."""
    df = df.reset_index() if df.index.name or not isinstance(df.index, pd.RangeIndex) else df
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        cells = [format(v, floatfmt) if isinstance(v, (float, np.floating)) else str(v) for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)
